# Smart Glass for Blind People — YOLOv3 Object Detection + Text-to-Speech

This notebook walks through the model architecture and detection pipeline used in the
**Smart Glass for Blind People Using Machine Learning** project.

**Pipeline:** Webcam image -> YOLOv3 (Darknet-53 backbone, COCO-trained) -> filter by
confidence -> grid-based position estimate (top/mid/bottom, left/center/right) ->
Google Text-to-Speech (gTTS) audio description.

> **Note:** This notebook loads the network **architecture** (`yolov3.cfg`) and class
> labels (`coco.names`), both included in this repo. The pretrained **weights**
> (`yolov3.weights`, ~236MB) are *not* included -- download them separately, see
> `models/README.md`, and place the file in the `models/` folder before running the
> detection cells below.


## 1. Imports

In [ ]:
import os
import time

import cv2
import numpy as np
from gtts import gTTS
from IPython.display import Audio, display


## 2. Model architecture (Darknet-53 / YOLOv3)

YOLOv3's backbone is **Darknet-53** -- 53 convolutional layers with residual
(skip) connections, borrowed from ResNet, so gradients propagate through the
deeper layers without vanishing. A 53-layer detection head is stacked on top,
giving **106 layers total**. Detections happen at **three scales**
(`13x13`, `26x26`, `52x52`) so both large and small objects are picked up --
this is the "multi-scale detector" described in the project report.

The full architecture is defined declaratively in `models/yolov3.cfg`
(committed to this repo -- it's a small text file, not a trained weight file).
Let's inspect it directly instead of hardcoding the architecture in Python:

In [ ]:
YOLO_DIR = "../models"  # adjust if running from a different working directory

cfg_path = os.path.join(YOLO_DIR, "yolov3.cfg")
names_path = os.path.join(YOLO_DIR, "coco.names")
weights_path = os.path.join(YOLO_DIR, "yolov3.weights")  # you must download this separately

with open(cfg_path) as f:
    cfg_lines = f.readlines()

# Count layer blocks by type, just to sanity check the architecture matches Darknet-53 + YOLO head
from collections import Counter
layer_types = [l.strip()[1:-1] for l in cfg_lines if l.startswith("[")]
print(f"Total layer blocks: {len(layer_types)}")
print(Counter(layer_types))


In [ ]:
with open(names_path) as f:
    LABELS = f.read().strip().split("\n")

print(f"Number of COCO classes: {len(LABELS)}")
print(LABELS[:10], "...")


## 3. Load the network

This step requires `yolov3.weights` to be present in `models/` -- see
`models/README.md` for the download link. Everything above this point
(architecture + labels) works without it.

In [ ]:
assert os.path.exists(weights_path), (
    f"{weights_path} not found. Download it first -- see models/README.md"
)

net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
print("YOLOv3 network loaded.")


## 4. Run detection on an image

In [ ]:
def detect_objects(net, image, confidence_thresh=0.5, nms_thresh=0.3):
    (H, W) = image.shape[:2]

    ln = net.getLayerNames()
    ln = [ln[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

    blob = cv2.dnn.blobFromImage(image, 1 / 255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)

    start = time.time()
    layer_outputs = net.forward(ln)
    end = time.time()
    print(f"YOLO forward pass took {end - start:.4f} seconds")

    boxes, confidences, class_ids = [], [], []

    for output in layer_outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > confidence_thresh:
                box = detection[0:4] * np.array([W, H, W, H])
                (center_x, center_y, width, height) = box.astype("int")
                x = int(center_x - (width / 2))
                y = int(center_y - (height / 2))

                boxes.append([x, y, int(width), int(height)])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, confidence_thresh, nms_thresh)
    return boxes, confidences, class_ids, idxs, (W, H)


In [ ]:
def describe_positions(boxes, class_ids, idxs, labels, frame_size):
    (W, H) = frame_size
    descriptions = []

    if len(idxs) == 0:
        return descriptions

    for i in idxs.flatten():
        (x, y) = (boxes[i][0], boxes[i][1])
        (w, h) = (boxes[i][2], boxes[i][3])

        center_x = round((2 * x + w) / 2)
        center_y = round((2 * y + h) / 2)

        w_pos = "left" if center_x <= W / 3 else "center" if center_x <= (W / 3 * 2) else "right"
        h_pos = "top" if center_y <= H / 3 else "mid" if center_y <= (H / 3 * 2) else "bottom"

        descriptions.append(f"{h_pos} {w_pos} {labels[class_ids[i]]}")

    return descriptions


In [ ]:
IMAGE_PATH = "../data/sample.jpg"  # replace with your own test image

image = cv2.imread(IMAGE_PATH)
assert image is not None, f"Could not read image at {IMAGE_PATH}"

boxes, confidences, class_ids, idxs, frame_size = detect_objects(net, image)
descriptions = describe_positions(boxes, class_ids, idxs, LABELS, frame_size)

description_text = ", ".join(descriptions) if descriptions else "no objects detected"
print(description_text)


## 5. Convert the description to speech (gTTS)

In [ ]:
if descriptions:
    tts = gTTS(text=description_text, lang="en", slow=False)
    tts.save("output.mp3")
    display(Audio("output.mp3"))
else:
    print("Nothing to speak -- no objects detected above the confidence threshold.")


## 6. Visualize detections (optional)

In [ ]:
import matplotlib.pyplot as plt

vis = image.copy()
if len(idxs) > 0:
    for i in idxs.flatten():
        (x, y, w, h) = boxes[i]
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(vis, LABELS[class_ids[i]], (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
